# IA supervisionada: classificação de crédito

Este notebook mostra um exemplo simples de IA supervisionada usando `scikit-learn`.

A ideia é prever se um cliente teria crédito `aprovado` ou `negado` com base em renda, idade, score de crédito e dívida atual.

## Problema

Fato: a base tem uma coluna chamada `decisao_credito`, que funciona como resposta correta.

Inferência: se um novo cliente for parecido com clientes aprovados no histórico, o modelo tende a prever aprovação.

Opinião técnica: KNN é uma boa escolha para aula porque é fácil explicar, o modelo compara clientes por similaridade.

In [ ]:
import os

os.environ.setdefault('LOKY_MAX_CPU_COUNT', '1')

import pandas as pd
import plotly.express as px
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler

pd.set_option('display.max_columns', None)

In [ ]:
def criar_base_credito() -> pd.DataFrame:
    dados_clientes = [
        {'renda_mensal': 3200, 'idade': 22, 'score_credito': 580, 'divida_atual': 2500, 'decisao_credito': 'negado'},
        {'renda_mensal': 7200, 'idade': 35, 'score_credito': 760, 'divida_atual': 1200, 'decisao_credito': 'aprovado'},
        {'renda_mensal': 4500, 'idade': 29, 'score_credito': 640, 'divida_atual': 3200, 'decisao_credito': 'negado'},
        {'renda_mensal': 8900, 'idade': 41, 'score_credito': 810, 'divida_atual': 900, 'decisao_credito': 'aprovado'},
        {'renda_mensal': 5100, 'idade': 31, 'score_credito': 690, 'divida_atual': 1800, 'decisao_credito': 'aprovado'},
        {'renda_mensal': 2700, 'idade': 24, 'score_credito': 520, 'divida_atual': 4200, 'decisao_credito': 'negado'},
        {'renda_mensal': 6300, 'idade': 38, 'score_credito': 710, 'divida_atual': 2100, 'decisao_credito': 'aprovado'},
        {'renda_mensal': 3900, 'idade': 27, 'score_credito': 610, 'divida_atual': 3500, 'decisao_credito': 'negado'},
        {'renda_mensal': 7600, 'idade': 45, 'score_credito': 780, 'divida_atual': 1500, 'decisao_credito': 'aprovado'},
        {'renda_mensal': 3400, 'idade': 36, 'score_credito': 590, 'divida_atual': 2800, 'decisao_credito': 'negado'},
        {'renda_mensal': 5800, 'idade': 33, 'score_credito': 700, 'divida_atual': 1600, 'decisao_credito': 'aprovado'},
        {'renda_mensal': 4200, 'idade': 25, 'score_credito': 600, 'divida_atual': 3900, 'decisao_credito': 'negado'},
        {'renda_mensal': 6800, 'idade': 30, 'score_credito': 730, 'divida_atual': 2200, 'decisao_credito': 'aprovado'},
        {'renda_mensal': 3100, 'idade': 28, 'score_credito': 550, 'divida_atual': 3600, 'decisao_credito': 'negado'},
        {'renda_mensal': 9300, 'idade': 49, 'score_credito': 830, 'divida_atual': 1100, 'decisao_credito': 'aprovado'},
        {'renda_mensal': 3700, 'idade': 39, 'score_credito': 570, 'divida_atual': 4100, 'decisao_credito': 'negado'},
        {'renda_mensal': 5600, 'idade': 26, 'score_credito': 680, 'divida_atual': 2400, 'decisao_credito': 'aprovado'},
        {'renda_mensal': 2500, 'idade': 21, 'score_credito': 500, 'divida_atual': 3800, 'decisao_credito': 'negado'},
    ]

    return pd.DataFrame(dados_clientes)

In [ ]:
def validar_base_credito(base_credito: pd.DataFrame) -> dict:
    colunas_entrada = ['renda_mensal', 'idade', 'score_credito', 'divida_atual']
    coluna_alvo = 'decisao_credito'
    rotulos_validos = {'aprovado', 'negado'}

    if base_credito.empty:
        raise ValueError('A base de crédito não pode estar vazia.')

    valores_nulos = base_credito.isna().sum().to_dict()
    if any(quantidade > 0 for quantidade in valores_nulos.values()):
        raise ValueError(f'Foram encontrados valores nulos: {valores_nulos}')

    for coluna in colunas_entrada:
        if not pd.api.types.is_numeric_dtype(base_credito[coluna]):
            raise TypeError(f'A coluna {coluna} precisa ser numérica.')

        if (base_credito[coluna] < 0).any():
            raise ValueError(f'A coluna {coluna} possui valor negativo.')

    rotulos_encontrados = set(base_credito[coluna_alvo].unique())
    if not rotulos_encontrados.issubset(rotulos_validos):
        raise ValueError(f'Rótulos inválidos encontrados: {rotulos_encontrados - rotulos_validos}')

    outliers_iqr = {}
    for coluna in colunas_entrada:
        primeiro_quartil = base_credito[coluna].quantile(0.25)
        terceiro_quartil = base_credito[coluna].quantile(0.75)
        intervalo_iqr = terceiro_quartil - primeiro_quartil
        limite_inferior = primeiro_quartil - 1.5 * intervalo_iqr
        limite_superior = terceiro_quartil + 1.5 * intervalo_iqr
        outliers_iqr[coluna] = int(((base_credito[coluna] < limite_inferior) | (base_credito[coluna] > limite_superior)).sum())

    return {
        'quantidade_linhas': len(base_credito),
        'valores_nulos': valores_nulos,
        'rotulos_encontrados': sorted(rotulos_encontrados),
        'outliers_iqr': outliers_iqr,
    }

In [ ]:
base_credito = criar_base_credito()
resumo_validacao_credito = validar_base_credito(base_credito)

display(base_credito)
resumo_validacao_credito

In [ ]:
def treinar_modelo_credito(base_credito: pd.DataFrame) -> dict:
    colunas_entrada = ['renda_mensal', 'idade', 'score_credito', 'divida_atual']
    coluna_alvo = 'decisao_credito'

    dados_entrada = base_credito[colunas_entrada]
    alvo_credito = base_credito[coluna_alvo]

    dados_treino, dados_teste, alvo_treino, alvo_teste = train_test_split(
        dados_entrada,
        alvo_credito,
        test_size=0.33,
        random_state=42,
        stratify=alvo_credito,
    )

    modelo_credito = Pipeline(
        steps=[
            ('normalizacao', MinMaxScaler()),
            ('classificador', KNeighborsClassifier(n_neighbors=3)),
        ]
    )
    modelo_credito.fit(dados_treino, alvo_treino)

    previsoes_credito = modelo_credito.predict(dados_teste)
    classes_credito = list(modelo_credito.named_steps['classificador'].classes_)
    matriz_confusao_credito = confusion_matrix(alvo_teste, previsoes_credito, labels=classes_credito)

    resultado_teste = dados_teste.copy()
    resultado_teste['real'] = alvo_teste.to_list()
    resultado_teste['previsto'] = previsoes_credito

    return {
        'modelo_credito': modelo_credito,
        'acuracia_credito': accuracy_score(alvo_teste, previsoes_credito),
        'classes_credito': classes_credito,
        'matriz_confusao_credito': matriz_confusao_credito,
        'resultado_teste': resultado_teste,
        'relatorio_classificacao': classification_report(
            alvo_teste,
            previsoes_credito,
            labels=classes_credito,
            output_dict=True,
            zero_division=0,
        ),
    }


resultado_credito = treinar_modelo_credito(base_credito)

print(f'Acurácia do modelo: {resultado_credito["acuracia_credito"]:.0%}')
display(resultado_credito['resultado_teste'])
display(pd.DataFrame(resultado_credito['relatorio_classificacao']).T.round(2))

In [ ]:
figura_clientes_credito = px.scatter(
    base_credito,
    x='renda_mensal',
    y='score_credito',
    color='decisao_credito',
    size='divida_atual',
    hover_data=['idade', 'divida_atual'],
    title='Clientes por renda, score e decisão de crédito',
    color_discrete_map={'aprovado': '#2f9e44', 'negado': '#d9480f'},
)
figura_clientes_credito.update_layout(template='plotly_white')
figura_clientes_credito.show()

In [ ]:
figura_matriz_confusao = px.imshow(
    resultado_credito['matriz_confusao_credito'],
    x=resultado_credito['classes_credito'],
    y=resultado_credito['classes_credito'],
    text_auto=True,
    labels={'x': 'Previsto', 'y': 'Real', 'color': 'Quantidade'},
    title='Matriz de confusão do KNN',
    color_continuous_scale='Blues',
)
figura_matriz_confusao.update_layout(template='plotly_white')
figura_matriz_confusao.show()

In [ ]:
def prever_cliente_credito(modelo_credito: Pipeline, dados_novo_cliente: dict) -> dict:
    novo_cliente = pd.DataFrame([dados_novo_cliente])
    classes_credito = list(modelo_credito.named_steps['classificador'].classes_)
    previsao_credito = modelo_credito.predict(novo_cliente)[0]
    probabilidades_credito = modelo_credito.predict_proba(novo_cliente)[0]

    return {
        'previsao_credito': previsao_credito,
        'probabilidades_credito': {
            classe: round(float(probabilidade), 3)
            for classe, probabilidade in zip(classes_credito, probabilidades_credito)
        },
    }


dados_novo_cliente = {
    'renda_mensal': 6200,
    'idade': 34,
    'score_credito': 705,
    'divida_atual': 1900,
}

resultado_novo_cliente = prever_cliente_credito(
    resultado_credito['modelo_credito'],
    dados_novo_cliente,
)

print('Novo cliente analisado:')
print(dados_novo_cliente)
print(f'Previsão: {resultado_novo_cliente["previsao_credito"]}')
print(f'Probabilidades: {resultado_novo_cliente["probabilidades_credito"]}')

## Como explicar em pouco tempo

Este é um caso supervisionado porque o modelo recebeu exemplos com resposta conhecida, `aprovado` ou `negado`.

O KNN compara o novo cliente com clientes antigos. Antes disso, o `MinMaxScaler` coloca renda, idade, score e dívida em escalas parecidas, porque KNN depende de distância.

Impacto prático: em uma empresa, isso poderia reduzir análise manual e ajudar a priorizar clientes, mas em produção seria obrigatório avaliar viés, risco de inadimplência, regras de negócio e auditoria.